In [1]:
import os
import scanpy as sc
import lamindb as ln
import zarr
import shutil
import pandas as pd

# --- 1. CONFIGURATION DES CHEMINS ---
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "data"))
H5AD_DIR = os.path.join(BASE_DIR, "script01_adatas")
ZARR_DIR = os.path.join(BASE_DIR, "script01_zarrs")

if not os.path.exists(ZARR_DIR):
    os.makedirs(ZARR_DIR)

def convert_h5ad_to_zarr(h5ad_filename):
    h5ad_path = os.path.join(H5AD_DIR, h5ad_filename)
    zarr_filename = h5ad_filename.replace(".h5ad", ".zarr")
    zarr_path = os.path.join(ZARR_DIR, zarr_filename)

    # Nettoyage pour éviter les conflits de version
    if os.path.exists(zarr_path):
        shutil.rmtree(zarr_path)

    print(f"⏳ Traitement de {h5ad_filename}...")
    try:
        adata = sc.read_h5ad(h5ad_path)
        
        # 1. Gestion des coordonnées spatiales
        # Vitessce cherche souvent 'X_spatial' dans obsm
        if 'x_centroid' in adata.obs.columns and 'y_centroid' in adata.obs.columns:
            adata.obsm['X_spatial'] = adata.obs[['x_centroid', 'y_centroid']].values
        
        # 2. Nettoyage des catégories (évite les erreurs de lecture JS)
        for col in adata.obs.columns:
            if adata.obs[col].dtype == 'category':
                adata.obs[col] = adata.obs[col].astype(str)

        # 3. Écriture forcée en ZARR V2 via DirectoryStore
        # C'est l'étape qui corrige le "NodeNotFoundError"
        store = zarr.DirectoryStore(zarr_path)
        adata.write_zarr(store)
        
        # 4. Consolidation des métadonnées
        # Vitessce a besoin du fichier .zmetadata pour scanner les dossiers
        zarr.consolidate_metadata(zarr_path)
        
        print(f"✅ Converti : {zarr_filename}")
        
    except Exception as e:
        print(f"❌ Erreur sur {h5ad_filename} : {e}")

# --- 2. EXÉCUTION ---
files = [f for f in os.listdir(H5AD_DIR) if f.endswith(".h5ad")]

if not files:
    print(f"⚠️ Aucun fichier .h5ad trouvé dans {H5AD_DIR}")
else:
    for f in files:
        convert_h5ad_to_zarr(f)

print("\n🚀 Terminé. Vide le cache de ton navigateur (Ctrl+F5) avant de tester le widget.")

⏳ Traitement de adata_OligoGr3_Core.h5ad...
✅ Converti : adata_OligoGr3_Core.zarr
⏳ Traitement de adata_AA2_Edge.h5ad...
✅ Converti : adata_AA2_Edge.zarr
⏳ Traitement de adata_AA1_Edge.h5ad...
✅ Converti : adata_AA1_Edge.zarr
⏳ Traitement de adata_GBM7-Edge.h5ad...
✅ Converti : adata_GBM7-Edge.zarr
⏳ Traitement de adata_Oligo2_Edge.h5ad...
✅ Converti : adata_Oligo2_Edge.zarr
⏳ Traitement de adata_GBM3-Core.h5ad...
✅ Converti : adata_GBM3-Core.zarr
⏳ Traitement de adata_OligoGr3_Edge.h5ad...
✅ Converti : adata_OligoGr3_Edge.zarr
⏳ Traitement de adata_GBM5-Core.h5ad...
✅ Converti : adata_GBM5-Core.zarr
⏳ Traitement de adata_Oligo5_Core.h5ad...
✅ Converti : adata_Oligo5_Core.zarr
⏳ Traitement de adata_GBM2-Core.h5ad...
✅ Converti : adata_GBM2-Core.zarr
⏳ Traitement de adata_AA1_Core.h5ad...
✅ Converti : adata_AA1_Core.zarr
⏳ Traitement de adata_GBM9-Edge.h5ad...
✅ Converti : adata_GBM9-Edge.zarr
⏳ Traitement de adata_Control_2.h5ad...
✅ Converti : adata_Control_2.zarr
⏳ Traitement de adat